# 1. Hugging Face Hub and Model Checkpoints

**Question: what exactly do we download when we download a model?**

In the previous chapter, we built mini GPT from its architecture code, configuration, tokenizer and trained `state_dict`. A Hub repository packages the equivalent artifacts for a pretrained model. In this notebook, we inspect each artifact explicitly before using Transformers for forward passes in the next notebook.

## 1. Environment and experiment record

First, we record the environment in which we run the experiment: Python and package versions, operating system, available device, memory and Hugging Face cache location. This gives us the context needed to reproduce the experiment and interpret differences on another machine.

We use the repository's `uv` environment. Qwen3 requires a Transformers version with support for the `qwen3` architecture, while `huggingface-hub` manages repository metadata and downloads, `safetensors` lets us inspect checkpoint tensors, and `psutil` reports system memory. Detecting MPS only tells us that the Apple GPU is available; it does not load the model or allocate its weights.

In [7]:
import json
import math
import platform
import shutil
import sys
from collections import Counter
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import psutil
import torch
from huggingface_hub import HfApi, hf_hub_download, snapshot_download
from huggingface_hub.constants import HF_HUB_CACHE
from IPython.display import Markdown, display
from safetensors import safe_open

MODEL_ID = "Qwen/Qwen3-0.6B-Base"
REQUESTED_REVISION = "main" 
CACHE_DIR = Path(HF_HUB_CACHE).expanduser()
SEED = 123
GiB = 1024 ** 3

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
memory = psutil.virtual_memory()
experiment = {
    "started_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version,
    "executable": sys.executable,
    "os": platform.platform(),
    "packages": {name: version(name) for name in ("torch", "transformers", "huggingface-hub", "safetensors", "psutil")},
    "available_device": device,
    "host_ram_total_GiB": memory.total / GiB,
    "host_ram_available_GiB": memory.available / GiB,
    "seed_for_later_experiments": SEED,
    "model_id": MODEL_ID,
    "requested_revision": REQUESTED_REVISION,
    "cache_dir": str(CACHE_DIR),
}
if device == "cuda":
    free, total = torch.cuda.mem_get_info()
    experiment["cuda"] = {"name": torch.cuda.get_device_name(), "free_GiB": free / GiB, "total_GiB": total / GiB}
print(json.dumps(experiment, indent=2))

{
  "started_utc": "2026-10-12T09:04:54.424904+00:00",
  "python": "3.12.13 (main, Jun  2 2026, 22:42:28) [Clang 22.1.3 ]",
  "executable": "/Users/sergiogomez/Repos/llm-lab/.venv/bin/python",
  "os": "macOS-26.6.2-arm64-arm-64bit",
  "packages": {
    "torch": "2.11.0",
    "transformers": "5.19.0",
    "huggingface-hub": "2.2.0",
    "safetensors": "0.8.0",
    "psutil": "7.2.2"
  },
  "available_device": "mps",
  "host_ram_total_GiB": 24.0,
  "host_ram_available_GiB": 6.5938262939453125,
  "seed_for_later_experiments": 123,
  "model_id": "Qwen/Qwen3-0.6B-Base",
  "requested_revision": "main",
  "cache_dir": "/Users/sergiogomez/.cache/huggingface/hub"
}


## 2. Explore the model repository

Next, we inspect `Qwen/Qwen3-0.6B-Base` before downloading its weights. We start with the base checkpoint because its pretraining objective is the closest match to our mini GPT; the post-trained checkpoint will be compared later.

The model card documents the intended use, architecture summary and Apache-2.0 license. The repository listing tells us which artifacts exist and how large they are without downloading their contents. We also resolve `main` to an immutable commit SHA immediately, so every later file comes from the same repository state.

In [8]:
api = HfApi()
info = api.model_info(MODEL_ID, revision=REQUESTED_REVISION, files_metadata=True)
REVISION = info.sha

assert REVISION, "The Hub did not return a commit revision."

experiment["resolved_revision"] = REVISION
repo_files = {item.rfilename: item for item in info.siblings}
card_data = info.card_data.to_dict() if info.card_data is not None else {}

print("Model:", MODEL_ID)
print("Commit:", REVISION)
print("License metadata:", card_data.get("license", "Not specified; inspect the license file"))
print("Pinned card:", f"https://huggingface.co/{MODEL_ID}/blob/{REVISION}/README.md")
for name, item in sorted(repo_files.items()):
    size = f"{item.size:,} bytes" if item.size is not None else "size unavailable"
    print(f"{name:48} {size}")

Model: Qwen/Qwen3-0.6B-Base
Commit: da87bfb608c14b7cf20ba1ce41287e8de496c0cd
License metadata: apache-2.0
Pinned card: https://huggingface.co/Qwen/Qwen3-0.6B-Base/blob/da87bfb608c14b7cf20ba1ce41287e8de496c0cd/README.md
.gitattributes                                   1,519 bytes
LICENSE                                          11,343 bytes
README.md                                        2,973 bytes
config.json                                      727 bytes
generation_config.json                           138 bytes
merges.txt                                       1,671,853 bytes
model.safetensors                                1,192,135,096 bytes
tokenizer.json                                   7,031,645 bytes
tokenizer_config.json                            9,678 bytes
vocab.json                                       2,776,833 bytes


## 3. Model IDs, revisions and local cache

We distinguish four related concepts:

- **Model ID:** the repository we select, such as `Qwen/Qwen3-0.6B-Base`.
- **Branch/tag:** a human-readable revision; `main` can move.
- **Commit SHA:** an immutable repository version that we pin for repeatability.
- **Cache:** local files reused across downloads. A snapshot path groups files by revision and may point to shared blobs. We treat cached files as read-only.

[`hf_hub_download`](https://huggingface.co/docs/huggingface_hub/en/guides/download) returns a local filename; `snapshot_download` returns a snapshot directory and can filter which files are fetched. A snapshot can be partial: a directory containing `config.json` is not necessarily a complete model. `local_files_only=True` forbids fetching missing files for that call; it does not make the whole notebook offline.

**Prediction:** fetching the same pinned config twice should reuse the same cached file. After resolving the revision once, we can save the printed SHA in `REQUESTED_REVISION` for future runs.

In [5]:
config_path = Path(hf_hub_download(
    repo_id=MODEL_ID, filename="config.json", revision=REVISION, cache_dir=CACHE_DIR,
))
cached_config_path = Path(hf_hub_download(
    repo_id=MODEL_ID, filename="config.json", revision=REVISION,
    cache_dir=CACHE_DIR, local_files_only=True,
))
assert config_path == cached_config_path
print("Snapshot file:", config_path)
print("Underlying file:", config_path.resolve())
print("Offline lookup reused the same path:", config_path == cached_config_path)
print("Pinned revision for future runs:", REVISION)

Snapshot file: /Users/sergiogomez/.cache/huggingface/hub/models--Qwen--Qwen3-0.6B-Base/snapshots/da87bfb608c14b7cf20ba1ce41287e8de496c0cd/config.json
Underlying file: /Users/sergiogomez/.cache/huggingface/hub/models--Qwen--Qwen3-0.6B-Base/blobs/43c79dcb3766612b23cbed17d0a56ce63efe4e74
Offline lookup reused the same path: True
Pinned revision for future runs: da87bfb608c14b7cf20ba1ce41287e8de496c0cd


## 4. Download configuration and tokenizer files

| File | Purpose | Learned neural-network weights? |
| --- | --- | --- |
| `config.json` | Architecture dimensions and settings | No |
| `tokenizer.json`, `vocab.json`, `merges.txt` | Text segmentation and token-to-ID mapping | No |
| `tokenizer_config.json`, special-token files | Tokenizer settings, special tokens, possibly a chat template | No |
| `generation_config.json` | Default generation settings, when present | No |
| `README.md`, license | Documentation and usage terms | No |
| `*.safetensors.index.json` | Tensor-name-to-shard mapping, when present | No |
| `*.safetensors` | Learned parameter tensors | Yes |

Now we download only the small metadata and tokenizer artifacts. A tokenizer vocabulary may itself be learned from data, but it does not contain the transformer's trained parameters. Configuration plus tokenizer cannot reproduce the pretrained model's predictions. We must use the tokenizer that matches the checkpoint because changing the meaning of an ID changes which embedding row is selected.

We use an explicit allowlist and intersect it with the pinned file listing; optional files need not exist.

In [6]:
metadata_candidates = {
    "config.json", "generation_config.json", "tokenizer.json", "tokenizer_config.json",
    "special_tokens_map.json", "added_tokens.json", "vocab.json", "merges.txt",
    "chat_template.jinja", "README.md", "LICENSE", "LICENSE.txt", "LICENSE.md",
}
metadata_files = sorted(metadata_candidates.intersection(repo_files))
snapshot_path = Path(snapshot_download(
    repo_id=MODEL_ID, revision=REVISION, cache_dir=CACHE_DIR,
    allow_patterns=metadata_files,
))
for name in metadata_files:
    print(f"{name:36} {(snapshot_path / name).stat().st_size:>12,} bytes")
if "README.md" in metadata_files:
    display(Markdown((snapshot_path / "README.md").read_text()))
for name in metadata_files:
    if name.startswith("LICENSE"):
        print((snapshot_path / name).read_text())

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

LICENSE                                    11,343 bytes
README.md                                   2,973 bytes
config.json                                   727 bytes
generation_config.json                        138 bytes
merges.txt                              1,671,853 bytes
tokenizer.json                          7,031,645 bytes
tokenizer_config.json                       9,678 bytes
vocab.json                              2,776,833 bytes


---
license: apache-2.0
library_name: transformers
pipeline_tag: text-generation
---
# Qwen3-0.6B-Base

## Qwen3 Highlights

Qwen3 is the latest generation of large language models in Qwen series, offering a comprehensive suite of dense and mixture-of-experts (MoE) models. 
Building upon extensive advancements in training data, model architecture, and optimization techniques, Qwen3 delivers the following key improvements over the previously released Qwen2.5:

- **Expanded Higher-Quality Pre-training Corpus:** Qwen3 is pre-trained on 36 trillion tokens across 119 languages — tripling the language coverage of Qwen2.5 — with a much richer mix of high-quality data, including coding, STEM, reasoning, book, multilingual, and synthetic data. 
- **Training Techniques and Model Architecture:** Qwen3 incorporates a series of training techiques and architectural refinements, including global-batch load balancing loss for MoE models and qk layernorm for all models, leading to improved stability and overall performance.
- **Three-stage Pre-training:** Stage 1 focuses on broad language modeling and general knowledge acquisition, Stage 2 improves reasoning skills like STEM, coding, and logical reasoning, and Stage 3 enhances long-context comprehension by extending training sequence lengths up to 32k tokens.
- **Scaling Law Guided Hyperparameter Tuning:** Through comprehensive scaling law studies across the three-stage pre-training pipeline, Qwen3 systematically tunes critical hyperparameters — such as learning rate scheduler and batch size — separately for dense and MoE models, resulting in better training dynamics and final performance across different model scales. 

## Model Overview

**Qwen3-0.6B-Base** has the following features:
- Type: Causal Language Models
- Training Stage: Pretraining
- Number of Parameters: 0.6B
- Number of Paramaters (Non-Embedding): 0.44B
- Number of Layers: 28
- Number of Attention Heads (GQA): 16 for Q and 8 for KV
- Context Length: 32,768 

For more details, including benchmark evaluation, hardware requirements, and inference performance, please refer to our [blog](https://qwenlm.github.io/blog/qwen3/), [GitHub](https://github.com/QwenLM/Qwen3), and [Documentation](https://qwen.readthedocs.io/en/latest/).

## Requirements

The code of Qwen3 has been in the latest Hugging Face `transformers` and we advise you to use the latest version of `transformers`.

With `transformers<4.51.0`, you will encounter the following error:
```
KeyError: 'qwen3'
```

## Evaluation & Performance

Detailed evaluation results are reported in this [📑 blog](https://qwenlm.github.io/blog/qwen3/).

### Citation

If you find our work helpful, feel free to give us a cite.

```
@misc{qwen3technicalreport,
      title={Qwen3 Technical Report}, 
      author={Qwen Team},
      year={2025},
      eprint={2505.09388},
      archivePrefix={arXiv},
      primaryClass={cs.CL},
      url={https://arxiv.org/abs/2505.09388}, 
}
```


                                 Apache License
                           Version 2.0, January 2004
                        http://www.apache.org/licenses/

   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION

   1. Definitions.

      "License" shall mean the terms and conditions for use, reproduction,
      and distribution as defined by Sections 1 through 9 of this document.

      "Licensor" shall mean the copyright owner or entity authorized by
      the copyright owner that is granting the License.

      "Legal Entity" shall mean the union of the acting entity and all
      other entities that control, are controlled by, or are under common
      control with that entity. For the purposes of this definition,
      "control" means (i) the power, direct or indirect, to cause the
      direction or management of such entity, whether by contract or
      otherwise, or (ii) ownership of fifty percent (50%) or more of the
      outstanding shares, or (iii) beneficial own

## 5. Read the model configuration

We now read `config.json` as the model's blueprint. The library's architecture code implements the operations, while checkpoint tensors supply their learned values. Reading this JSON creates neither a model nor its weights. `transformers_version` describes the saved artifact's provenance, not the version currently installed in our environment.

| Qwen config field | Mini GPT counterpart / meaning |
| --- | --- |
| `vocab_size` | `vocab_size`: number of embedding/output rows |
| `hidden_size` | `emb_dim`: residual-stream width |
| `num_hidden_layers` | `n_layers`: repeated transformer blocks |
| `num_attention_heads` | `n_heads`: query heads |
| `max_position_embeddings` | Similar role to `context_length`, but positional mechanism differs |
| `intermediate_size` | Feed-forward expansion width; mini GPT used `4 * emb_dim` |
| `num_key_value_heads` | GQA: several query heads share key/value heads |
| `head_dim` | Explicit width of each attention head |
| `rope_theta`, `rope_scaling` | RoPE rotates queries/keys using token position |
| `rms_norm_eps` | RMSNorm normalizes using root mean square, without subtracting the mean |
| `hidden_act` | SiLU participates in Qwen's gated SwiGLU feed-forward network |
| `tie_word_embeddings` | Input embeddings and output projection share parameters |

**Important difference:** we cannot assume `head_dim = hidden_size / num_attention_heads`. In this Qwen configuration, the explicit head dimension makes the concatenated query width differ from the residual width, and a projection brings it back. With PyTorch linear weights stored as `[out_features, in_features]`, an input `[batch, tokens, hidden_size]` produces queries `[batch, tokens, query_heads * head_dim]`, then reshapes to `[batch, query_heads, tokens, head_dim]`.

We only identify RoPE, RMSNorm, SwiGLU and grouped-query attention (GQA) here; notebook 02 will connect them to a real forward pass. A configured context limit does not guarantee that the longest sequence fits in our available memory.

Source: [Qwen base configuration](https://huggingface.co/Qwen/Qwen3-0.6B-Base/blob/main/config.json). The cell below reads the version pinned by our resolved commit.

In [ ]:
config = json.loads(config_path.read_text())
fields = [
    "model_type", "architectures", "vocab_size", "hidden_size", "intermediate_size",
    "num_hidden_layers", "num_attention_heads", "num_key_value_heads", "head_dim",
    "max_position_embeddings", "rope_theta", "rope_scaling", "rms_norm_eps",
    "hidden_act", "tie_word_embeddings", "torch_dtype", "dtype", "transformers_version",
]

for field in fields:
    print(f"{field:28} {config.get(field, '(not stored)')}")
    
H = config["hidden_size"]
Q = config["num_attention_heads"]
KV = config["num_key_value_heads"]
D = config["head_dim"]

assert Q % KV == 0

expected_shapes = {
    "model.embed_tokens.weight": (config["vocab_size"], H),
    "model.layers.0.self_attn.q_proj.weight": (Q * D, H),
    "model.layers.0.self_attn.k_proj.weight": (KV * D, H),
    "model.layers.0.self_attn.v_proj.weight": (KV * D, H),
    "model.layers.0.self_attn.o_proj.weight": (H, Q * D),
    "model.layers.0.mlp.gate_proj.weight": (config["intermediate_size"], H),
}

print("\nPredicted tensor shapes (to verify after download):")

for name, shape in expected_shapes.items():
    print(name, shape)
print("Query heads per KV head:", Q // KV)

model_type                   qwen3
architectures                ['Qwen3ForCausalLM']
vocab_size                   151936
hidden_size                  1024
intermediate_size            3072
num_hidden_layers            28
num_attention_heads          16
num_key_value_heads          8
head_dim                     128
max_position_embeddings      32768
rope_theta                   1000000
rope_scaling                 None
rms_norm_eps                 1e-06
hidden_act                   silu
tie_word_embeddings          True
torch_dtype                  bfloat16
dtype                        (not stored)
transformers_version         4.51.0

Predicted tensor shapes (to verify after download):
model.embed_tokens.weight (151936, 1024)
model.layers.0.self_attn.q_proj.weight (2048, 1024)
model.layers.0.self_attn.k_proj.weight (1024, 1024)
model.layers.0.self_attn.v_proj.weight (1024, 1024)
model.layers.0.self_attn.o_proj.weight (1024, 2048)
model.layers.0.mlp.gate_proj.weight (3072, 1024)
Query h

## 6. Inspect the checkpoint format

We use **checkpoint** here to mean a saved collection of model parameter tensors, analogous to our mini GPT `state_dict`. It is not necessarily a full training-resume checkpoint containing optimizer and scheduler state.

[Safetensors](https://huggingface.co/docs/safetensors/en/index) stores named tensors with shapes and dtypes. Large checkpoints may be split into **shards**; an index maps each tensor name to its file. Sharding changes storage layout, not model architecture. This small model may have only one weight file and no index; the code handles both.

Before downloading the weights, we inspect their filenames and any small index. After downloading them, we use `safe_open` metadata access to check the actual tensor shapes without constructing a Transformers model or materializing every tensor. Tensor names connect saved arrays to modules, shapes specify their dimensions, and dtypes specify their numerical representation.

In [10]:
index_name = "model.safetensors.index.json"
weight_index = None

if index_name in repo_files:
    index_path = Path(hf_hub_download(
        repo_id=MODEL_ID, filename=index_name, revision=REVISION, cache_dir=CACHE_DIR,
    ))
    weight_index = json.loads(index_path.read_text())
    weight_files = sorted(set(weight_index["weight_map"].values()))
    print("Index metadata:", weight_index.get("metadata", {}))
    print("Example tensor-to-shard entries:")
    for name, shard in list(weight_index["weight_map"].items())[:8]:
        print(name, "->", shard)
elif "model.safetensors" in repo_files:
    weight_files = ["model.safetensors"]
    print("Single weight file; no shard index needed.")
else:
    raise ValueError("Expected a standard safetensors checkpoint; inspect the repository listing.")

assert all(name in repo_files for name in weight_files)

print("Weight files:", weight_files)
sizes = [repo_files[name].size for name in weight_files]
remote_weight_bytes = sum(sizes) if all(size is not None for size in sizes) else None
print("Repository-reported weight bytes:", remote_weight_bytes)

Single weight file; no shard index needed.
Weight files: ['model.safetensors']
Repository-reported weight bytes: 1192135096


## 7. Download the model weights

This is the large-download step. We reuse any pinned files already present in the cache, and we still do not construct a model or load weights into accelerator memory.

For an approximate parameter count $P$ and bytes per parameter $b$, weight memory is $P b$. Using the rounded 0.6-billion label gives about 1.2 GB at BF16/FP16 (2 bytes) or 2.4 GB at FP32 (4 bytes). GB is $10^9$ bytes; GiB is $2^{30}$ bytes. The next cell compares these estimates with repository file sizes and then the actual downloaded tensors.

These are three different quantities: **download/storage size**, **weight memory in the chosen runtime dtype**, and **total inference memory**. Inference also needs activations, attention KV cache and runtime workspace. A download fitting on disk does not establish that inference will fit in memory. Available RAM is only a snapshot, not a reservation.

**Predict before running:** should BF16 checkpoint tensor bytes be closer to the 2-byte or 4-byte estimate? Will an output projection need separate stored weights when embeddings are tied?

In [11]:
approx_parameters = 600_000_000  # Rounded model label, not an exact count.
for dtype_name, bytes_per_parameter in [("BF16 / FP16", 2), ("FP32", 4)]:
    print(f"{dtype_name}: estimated weights = {approx_parameters * bytes_per_parameter / GiB:.3f} GiB")
free_disk = shutil.disk_usage(snapshot_path).free
print(f"Free disk at cache: {free_disk / GiB:.3f} GiB")
if remote_weight_bytes is not None:
    print(f"Published weight files: {remote_weight_bytes / GiB:.3f} GiB")
    print("Network transfer may be smaller if some or all files are already cached.")

snapshot_path = Path(snapshot_download(
    repo_id=MODEL_ID, revision=REVISION, cache_dir=CACHE_DIR,
    allow_patterns=weight_files,
))
tensor_records = {}
for filename in weight_files:
    weight_path = snapshot_path / filename
    print(f"{filename}: {weight_path.stat().st_size:,} bytes")
    with safe_open(str(weight_path), framework="pt", device="cpu") as weights:
        for name in weights.keys():
            tensor_slice = weights.get_slice(name)
            assert name not in tensor_records, f"Duplicate tensor: {name}"
            tensor_records[name] = {
                "shape": tuple(tensor_slice.get_shape()),
                "dtype": tensor_slice.get_dtype(),
                "file": filename,
            }
for name, expected in expected_shapes.items():
    actual = tensor_records[name]
    print(name, actual)
    assert actual["shape"] == expected, (name, actual["shape"], expected)
if weight_index is not None:
    assert set(tensor_records) == set(weight_index["weight_map"])
    assert all(row["file"] == weight_index["weight_map"][name] for name, row in tensor_records.items())

stored_elements = sum(math.prod(row["shape"]) for row in tensor_records.values())
dtype_counts = Counter(row["dtype"] for row in tensor_records.values())
print("Stored tensor elements:", f"{stored_elements:,}")
print("Tensor count by dtype:", dict(dtype_counts))
print("Separate lm_head.weight stored:", "lm_head.weight" in tensor_records)
print("Configured tied embeddings:", config["tie_word_embeddings"])
# Tied parameters may be serialized once; do not count a missing tied lm_head twice.
bytes_by_dtype = {"BF16": 2, "F16": 2, "F32": 4, "F64": 8}
if all(row["dtype"] in bytes_by_dtype for row in tensor_records.values()):
    tensor_bytes = sum(math.prod(row["shape"]) * bytes_by_dtype[row["dtype"]] for row in tensor_records.values())
    print(f"Exact stored tensor payload: {tensor_bytes / GiB:.6f} GiB")
    print("File size also includes tensor headers.")

BF16 / FP16: estimated weights = 1.118 GiB
FP32: estimated weights = 2.235 GiB
Free disk at cache: 287.105 GiB
Published weight files: 1.110 GiB
Network transfer may be smaller if some or all files are already cached.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors: 1,192,135,096 bytes
model.embed_tokens.weight {'shape': (151936, 1024), 'dtype': 'BF16', 'file': 'model.safetensors'}
model.layers.0.self_attn.q_proj.weight {'shape': (2048, 1024), 'dtype': 'BF16', 'file': 'model.safetensors'}
model.layers.0.self_attn.k_proj.weight {'shape': (1024, 1024), 'dtype': 'BF16', 'file': 'model.safetensors'}
model.layers.0.self_attn.v_proj.weight {'shape': (1024, 1024), 'dtype': 'BF16', 'file': 'model.safetensors'}
model.layers.0.self_attn.o_proj.weight {'shape': (1024, 2048), 'dtype': 'BF16', 'file': 'model.safetensors'}
model.layers.0.mlp.gate_proj.weight {'shape': (3072, 1024), 'dtype': 'BF16', 'file': 'model.safetensors'}
Stored tensor elements: 596,049,920
Tensor count by dtype: {'BF16': 310}
Separate lm_head.weight stored: False
Configured tied embeddings: True
Exact stored tensor payload: 1.110229 GiB
File size also includes tensor headers.


## 8. Checkpoint inventory and conclusions

Finally, we record local file sizes, the exact revision, software versions and shape checks. We verify that every selected file can be located with network access disabled. The cell prints the experiment record directly in the notebook, so saving the notebook also preserves the observed evidence.

To reproduce these artifacts, we copy the resolved SHA into `REQUESTED_REVISION`, use the recorded dependency environment and repeat the same downloads. The revision pins the model files, but it does not pin Python libraries or guarantee identical numerical results across hardware.

**Expected lesson, not a measured result:** the Hub provides the same ingredients we used in mini GPT in a standardized layout. Downloading saves those ingredients on disk, model construction allocates modules, weight loading fills their parameters, and inference computes outputs. These are separate steps.

In [12]:
def file_purpose(name):
    if name.endswith(".safetensors"):
        return "learned tensor weights"
    if name.endswith(".index.json"):
        return "tensor-to-shard map"
    if name == "config.json":
        return "architecture configuration"
    if name == "generation_config.json":
        return "generation defaults"
    if name == "README.md" or name.startswith("LICENSE"):
        return "documentation / license"
    return "tokenizer / text formatting"

selected_files = sorted(set(metadata_files + weight_files + ([index_name] if weight_index else [])))
inventory = []
for name in selected_files:
    path = Path(hf_hub_download(
        repo_id=MODEL_ID, filename=name, revision=REVISION,
        cache_dir=CACHE_DIR, local_files_only=True,
    ))
    inventory.append({
        "filename": name,
        "purpose": file_purpose(name),
        "bytes": path.stat().st_size,
        "learned_tensors": name.endswith(".safetensors"),
    })
experiment.update({
    "snapshot_path": str(snapshot_path),
    "inventory": inventory,
    "stored_tensor_elements": stored_elements,
    "tensor_count_by_dtype": dict(dtype_counts),
    "weight_file_bytes": sum((snapshot_path / name).stat().st_size for name in weight_files),
    "predicted_shapes_verified": list(expected_shapes),
    "selected_files_available_offline": True,
    "model_loaded": False,
    "inference_run": False,
})
print(json.dumps(experiment, indent=2))

{
  "started_utc": "2026-10-12T09:04:54.424904+00:00",
  "python": "3.12.13 (main, Jun  2 2026, 22:42:28) [Clang 22.1.3 ]",
  "executable": "/Users/sergiogomez/Repos/llm-lab/.venv/bin/python",
  "os": "macOS-26.6.2-arm64-arm-64bit",
  "packages": {
    "torch": "2.11.0",
    "transformers": "5.19.0",
    "huggingface-hub": "2.2.0",
    "safetensors": "0.8.0",
    "psutil": "7.2.2"
  },
  "available_device": "mps",
  "host_ram_total_GiB": 24.0,
  "host_ram_available_GiB": 6.5938262939453125,
  "seed_for_later_experiments": 123,
  "model_id": "Qwen/Qwen3-0.6B-Base",
  "requested_revision": "main",
  "cache_dir": "/Users/sergiogomez/.cache/huggingface/hub",
  "resolved_revision": "da87bfb608c14b7cf20ba1ce41287e8de496c0cd",
  "snapshot_path": "/Users/sergiogomez/.cache/huggingface/hub/models--Qwen--Qwen3-0.6B-Base/snapshots/da87bfb608c14b7cf20ba1ce41287e8de496c0cd",
  "inventory": [
    {
      "filename": "LICENSE",
      "purpose": "documentation / license",
      "bytes": 11343,
      "

## Conclusions

We inspected `Qwen/Qwen3-0.6B-Base`, resolved `main` to the immutable commit `da87bfb608c14b7cf20ba1ce41287e8de496c0cd`, and downloaded its artifacts into the local Hugging Face cache. We separated documentation, configuration and tokenizer files from the learned weights, then inspected the single `model.safetensors` checkpoint. It contains 596,049,920 stored elements across 310 BF16 tensors, with a tensor payload of 1.110229 GiB. The real embedding, attention and MLP tensor shapes matched the shapes predicted from `config.json`, and the absence of a separate `lm_head.weight` confirmed that the input and output embeddings are tied.

This experiment connects a Hub checkpoint with the components of our mini GPT: architecture code defines the operations, `config.json` defines their dimensions, tokenizer files map text to token IDs, and Safetensors stores the trained parameter values. Downloading these artifacts only places them persistently on disk; it does not construct the PyTorch model, load its weights into CPU or MPS memory, or run inference. The next notebook will load the tokenizer and model and follow the path from text to token IDs and logits.